# Catálogo de Itens de Ciências da Natureza (ENEM 2024)

## 1. Configuração

In [ ]:
import polars as pl
from pathlib import Path

# Caminhos principais do projeto
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
RAW = ROOT / "data" / "raw"
INDICADORES = ROOT / "data" / "processed" / "indicadores"
ENEM = ROOT / "data" / "processed" / "enem"

ENEM.mkdir(parents=True, exist_ok=True)

## 2. Mapeamento Disciplinar via CO_ITEM

In [ ]:
# Lê as planilhas de provas (Amarela, Azul, Verde) e remove linhas vazias
provas = pl.concat([
    pl.read_excel(p, engine="calamine").drop_nulls("CO_PROVA")
    for p in sorted((RAW / "Provas").glob("*.xlsx"))
]).with_columns(
    pl.col("CO_PROVA", "CO_POSICAO", "IN_FISICA", "IN_BIOLOGIA", "IN_QUIMICA").cast(pl.Int64)
)

# Lê os itens oficiais do ENEM 2024 com parâmetros TRI
itens = pl.read_csv(
    RAW / "microdados_enem_2024" / "ITENS_PROVA_2024.csv",
    separator=";", encoding="iso-8859-1"
)

# Identifica a disciplina de cada código único de questão (CO_ITEM)
mapa_item = (
    provas.join(itens, on=["CO_PROVA", "CO_POSICAO"], how="inner")
    .select("CO_ITEM", "IN_FISICA", "IN_BIOLOGIA", "IN_QUIMICA")
    .unique()
)
print("Itens únicos de CN classificados:", mapa_item.height)

## 3. Propagação para Todos os Cadernos de CN

In [ ]:
# Propaga a disciplina pelo CO_ITEM para cobrir também o caderno Cinza (1422) e adaptados
itens_cn = (
    itens.filter(pl.col("SG_AREA") == "CN")
    .join(mapa_item, on="CO_ITEM", how="left")
    .with_columns(
        pl.when(pl.col("IN_FISICA") == 1).then(pl.lit("Física"))
        .when(pl.col("IN_QUIMICA") == 1).then(pl.lit("Química"))
        .when(pl.col("IN_BIOLOGIA") == 1).then(pl.lit("Biologia"))
        .otherwise(None)
        .alias("DISCIPLINA"),
        # Índice de 0 a 44 na string TX_RESPOSTAS_CN (questões 91 a 135)
        (pl.col("CO_POSICAO") - 91).alias("IDX_POSICAO"),
    )
    .sort(["CO_PROVA", "CO_POSICAO"])
)

itens_cn.write_parquet(ENEM / "itens_cn_2024.parquet")
print("Catálogo salvo:", itens_cn.shape)
itens_cn.filter(pl.col("CO_PROVA") == 1420).head(10)